# PhD thesis figures (Nicholas Jannsen)

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
import os
import h5py
import scipy
import numpy as np
import pandas as pd
from tqdm import tqdm 

from matplotlib import pyplot as plt
import matplotlib.colors  as colors
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable

from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord

from scipy.signal import periodogram
from scipy.ndimage import median_filter
from scipy.interpolate import interp2d

import transitleastsquares as tls

# PlatoSim
import platosim.referenceFrames as rf
import platosim.plot            as pt
import platosim.utilities       as ut
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.lightcurve   import LightCurve
from platosim.validation import switchOffAllEffects
from platosim.matplotlibrc import setup_paper
setup_paper()

# Constants
day = 86400

In [ ]:
# Save figure
path = '/lhome/nicholas/Nextcloud/thesis'
outputDir = os.getcwd()

## Figures for cover

In [ ]:
star = 'SPB'
pathx = f'{path}/{star}/clean'
patha = f'{path}/{star}/affogato'
pathc = f'{path}/{star}/cortado'
pathd = f'{path}/{star}/doppio'
pathv = f'{path}/{star}/varsource'

In [ ]:
# Choose simulation
idir = patha
starID = f'{3}'.zfill(9)

# Fetch final ligth curve
lc = LightCurve(f'{idir}/lightcurve/lc_{starID}.ftr', mode="final")
df = lc.data()

# Load sim table
dt = pd.read_feather(f'{idir}/table/table_{starID}.ftr')

# Create varsource from pulsations
dx = pd.read_feather(f'{pathv}/pulsations/pulsations_{starID}_001.ftr')
dv = pd.DataFrame()
dv['time'] = df.time / 86400
dv['dmag'] = ns.timeSeriesFromFourier(dv.time, dx.freq, dx.ampl, dx.phase, power=2.2)
dv['flux'] = (10**(-0.4*dv.dmag) - 1) * 1e6

In [ ]:
fig = plt.figure(figsize=(6,1.2))
plt.plot(dv.time, dv.flux/100, '-', c='w', lw=1.5)
plt.xlim(0, 20)
plt.xticks([])
plt.yticks([])
plt.axis('off')
plt.ylim(-13, 7)
plt.tight_layout()
plt.show()

# Save figure
tdir = '/lhome/nicholas/Nextcloud/thesis/image'
fig.savefig(f'{tdir}/lightcurve_gmode.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# python varsim.py --star Sun --planet hot-Neptune --seed 9
ds = pd.read_csv(f'{path}/../plot/varsource_sunlike.txt', sep=' ', names=['time', 'dmag'])
time = ds.time / 86400
flux = 10**(-ds.dmag/2.5)

fig = plt.figure(figsize=(6.6,1.5))
plt.plot(time, flux, '-', c='w', lw=0.5)
plt.xlim(3, 60)
plt.xticks([])
plt.yticks([])
plt.axis('off')
# plt.ylim(-13, 7)
plt.tight_layout()
plt.show()

# Save figure
tdir = '/lhome/nicholas/Nextcloud/thesis/image'
fig.savefig(f'{tdir}/lightcurve_sunlike.png', bbox_inches='tight', dpi=300, transparent=True)

## Figure for Sect. 1.5: Photometry

In [ ]:
# Initialise PlatoSim (we reuse the inputfile.yaml)
outputFileName = "output_subfield"
sim = Simulation(outputFileName, outputDir=os.getcwd())

# Obs parameters
sim["ObservingParameters/NumExposures"] = 1
# sim["Platform/SolarPanelOrientation"] = 0

# Sky parameters (auto background)
sim["Sky/SkyBackground/UseConstantSkyBackground"] = 'yes'
sim["Sky/SkyBackground/BackgroundValue"]          = -1 
sim["Sky/Cosmics/CosmicHitRate"] = 10

# Subfield
rows, cols = 300, 300
# rows, cols = 1500, 800
sim["SubField/NumColumns"]      = 500
sim["SubField/NumRows"]         = 500
sim["SubField/ZeroPointColumn"] = cols
sim["SubField/ZeroPointRow"]    = rows

# Run simulation
simfile0 = sim.run(removeOutputFile=True)

In [ ]:
# Show image
f0 = SimFile(outputFileName + ".hdf5")
fig, ax = f0.showImage(0, clip=1.0, figsize=(5,5),
                       colorMap="cubehelix", colorBar=False)
ax.set_xticks([])
ax.set_yticks([])
ax.axis('off')

# Save figure
fig.savefig(f'{path}/chapters/introduction/image.png', bbox_inches='tight', dpi=300)

In [ ]:
# Find target from ,catalogue of Image
# PSF photometry star ID 10
# Aperture photometry star ID 4
cat = simfile0.getStarCoordinates(0)
starID = 10
rowStar = int(cat[1][starID])
colStar = int(cat[2][starID])
fluxStar = int(cat[-1][starID])
rowStar, colStar

In [ ]:
# Initialise PlatoSim
outputFile = "output_imagette"
sim = Simulation(outputFile, outputDir=os.getcwd())

# Observation
sim["ObservingParameters/NumExposures"] = 1

# Sky
sim["Sky/SkyBackground/UseConstantSkyBackground"] = 'yes'
sim["Sky/SkyBackground/BackgroundValue"]          = -1 
sim["Sky/Cosmics/CosmicHitRate"] = 10

# Subfield
sim["SubField/NumColumns"]      = 6
sim["SubField/NumRows"]         = 6
sim["SubField/ZeroPointColumn"] = cols + colStar - 2
sim["SubField/ZeroPointRow"]    = rows + rowStar - 3

# Control HDF5
sim["ControlHDF5Content/WriteStarPositions"]     = True
sim["ControlHDF5Content/WriteDiffusedPSF"]       = True
sim["ControlHDF5Content/WriteHighResolutionPSF"] = True

# Run simulation
simfile = sim.run(removeOutputFile=True)

In [ ]:
# Show image
f = SimFile(outputFile + ".hdf5")
fig, ax = f.showImage(0, figsize=(5,5), clip=1.0, 
                      colorMap="cubehelix", colorBar=False,
                      tarMarkerSize=500) 

ax.set_xticks([])
ax.set_yticks([])
ax.axis('off')

# Save figure
fig.savefig(f'{path}/chapters/introduction/imagette_aperture.png', bbox_inches='tight', dpi=300)

### Aperture photometry of contaminated target

In [ ]:
# Set up a Simulation object
sim = Simulation("output_example3", outputDir=outputDir)

# Load variable source
starID = [0, 1]
inputDir = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
variableSourceFile0 = inputDir + "/varsource_gdor.txt"
variableSourceFile1 = inputDir + "/varsource_algol.txt"
variableSourceFiles = [variableSourceFile0, variableSourceFile1]

# Create and set variable source file
variableSourceList  = outputDir + "/varlist_example3.txt"
sim.createVariableSourceList(starID, variableSourceFiles, variableSourceList)

# Create and set photometry file (for target star only)
starID = [0]
photometryFile = outputDir + "/photometry_example1.txt"
sim.createPhotometryFile(starID, photometryFile)

# Select subfield size and location
sim["SubField/NumColumns"]      = 8
sim["SubField/NumRows"]         = 8
sim["SubField/ZeroPointRow"]    = 3000
sim["SubField/ZeroPointColumn"] = 3000

# Define catalogue
row = np.array([4.0, 3.1]) + sim["SubField/ZeroPointRow"]
col = np.array([4.0, 2.8]) + sim["SubField/ZeroPointColumn"]
mag = np.array([12, 14])
starID = [0, 1]

# Create and set stellar catalogue file
starcatFile = outputDir + "/starcat_example3.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, starID, starcatFile)

# Save only the needed output
sim.turnOffAllOutput()
sim["ControlHDF5Content/GroupByExposure"]    = True
sim["ControlHDF5Content/WritePixelMaps"]     = True
sim["ControlHDF5Content/WriteStarPositions"] = True
sim["ControlHDF5Content/WriteCosmics"]       = True

# Run simulation
dt = 600
tdur = 30
nexp = int(tdur * 86400/dt)
sim["ObservingParameters/CycleTime"] = dt
sim["ObservingParameters/NumExposures"] = nexp

# Run simulation
simfile = sim.run(removeOutputFile=True, executionTime=True)

In [ ]:
f = SimFile("output_example3.hdf5")
df = f.getLightCurve(starID[0], fluxType='estimated', df=True)

fig, ax = plt.subplots(1, 1, figsize=(8,3))
ax.plot(df.time/86400, ut.normalize(df.flux, factor=1e3), '.', c='fuchsia', ms=5, alpha=0.2)
ax.axvline(x=14, linestyle=':', c='k')
ax.set_xlim(0, tdur)
ax.set_xlabel(r'Time $\rightarrow$')
ax.set_ylabel(r'Flux $\rightarrow$')
ax.set_xticks([])
ax.set_yticks([]);

# Save figure
fig.savefig(f'{path}/chapters/introduction/photometry_aperture.png', bbox_inches='tight', dpi=300)

### PSF photometry target

In [ ]:
# Set up a Simulation object
sim = Simulation("output_psf_target", outputDir=outputDir)

# Run simulation
dt = 600
tdur = 30
nexp = int(tdur * 86400/dt)
sim["ObservingParameters/CycleTime"] = dt
sim["ObservingParameters/NumExposures"] = nexp

# Load variable source
starID = [0]
inputDir = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
variableSourceFile0 = inputDir + "/varsource_gdor.txt"
variableSourceFiles = [variableSourceFile0]

# Create and set variable source file
variableSourceList  = outputDir + "/varlist.txt"
sim.createVariableSourceList(starID, variableSourceFiles, variableSourceList)

# Create and set photometry file (for target star only)
starID = [0]
photometryFile = outputDir + "/photometry.txt"
sim.createPhotometryFile(starID, photometryFile)

# Change mask update
sim["Photometry/MaskUpdateInterval"] = tdur

# Select subfield size and location
sim["SubField/NumColumns"]      = 8
sim["SubField/NumRows"]         = 8
sim["SubField/ZeroPointRow"]    = 3000
sim["SubField/ZeroPointColumn"] = 3000

# Define catalogue
row = np.array([4.0, 3.1]) + sim["SubField/ZeroPointRow"]
col = np.array([4.0, 2.8]) + sim["SubField/ZeroPointColumn"]
mag = np.array([12, 14])
starID = [0, 1]

# Create and set stellar catalogue file
starcatFile = outputDir + "/starcat.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, starID, starcatFile)

# Save only the needed output
sim.turnOffAllOutput()
sim["ControlHDF5Content/GroupByExposure"]    = True
sim["ControlHDF5Content/WritePixelMaps"]     = True
sim["ControlHDF5Content/WriteStarPositions"] = True
sim["ControlHDF5Content/WriteCosmics"]       = True

# Run simulation
simfile = sim.run(removeOutputFile=True, executionTime=True)

In [ ]:
# Set up a Simulation object
sim = Simulation("output_psf_contaminant", outputDir=outputDir)

# Run simulation
dt = 600
tdur = 30
nexp = int(tdur * 86400/dt)
sim["ObservingParameters/CycleTime"] = dt
sim["ObservingParameters/NumExposures"] = nexp

# Load variable source
starID = [1]
inputDir = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
variableSourceFile1 = inputDir + "/varsource_algol.txt"
variableSourceFiles = [variableSourceFile1]

# Create and set variable source file
variableSourceList  = outputDir + "/varlist.txt"
sim.createVariableSourceList(starID, variableSourceFiles, variableSourceList)

# Create and set photometry file (for target star only)
starID = [1]
photometryFile = outputDir + "/photometry.txt"
sim.createPhotometryFile(starID, photometryFile)

# Change mask update
sim["Photometry/MaskUpdateInterval"] = tdur

# Select subfield size and location
sim["SubField/NumColumns"]      = 8
sim["SubField/NumRows"]         = 8
sim["SubField/ZeroPointRow"]    = 3000
sim["SubField/ZeroPointColumn"] = 3000

# Define catalogue
row = np.array([4.0, 3.1]) + sim["SubField/ZeroPointRow"]
col = np.array([4.0, 2.8]) + sim["SubField/ZeroPointColumn"]
mag = np.array([12, 14])
starID = [0, 1]

# Create and set stellar catalogue file
starcatFile = outputDir + "/starcat.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, starID, starcatFile)

# Save only the needed output
sim.turnOffAllOutput()
sim["ControlHDF5Content/GroupByExposure"]    = True
sim["ControlHDF5Content/WritePixelMaps"]     = True
sim["ControlHDF5Content/WriteStarPositions"] = True
sim["ControlHDF5Content/WriteCosmics"]       = True

# Run simulation
simfile = sim.run(removeOutputFile=True, executionTime=True)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,3))

f = SimFile("output_psf_target.hdf5")
df = f.getLightCurve(starID[0], fluxType='estimated', df=True)
ax.plot(df.time/86400, ut.normalize(df.flux, factor=1e3), '.', c='fuchsia', ms=5, alpha=0.2)

f = SimFile("output_psf_contaminant.hdf5")
df = f.getLightCurve(starID[1], fluxType='estimated', df=True)
ax.plot(df.time/86400, ut.normalize(df.flux, factor=1e3)/4-70, '.', c='limegreen', ms=5, alpha=0.2)

ax.set_xlim(0, tdur)
ax.set_xlabel(r'Time $\rightarrow$')
ax.set_ylabel(r'Flux $\rightarrow$')
ax.set_xticks([])
ax.set_yticks([]);

# Save figure
fig.savefig(f'{path}/chapters/introduction/photometry_psf.png', bbox_inches='tight', dpi=300)

---
## AGNs and SMBHBs
---

In [ ]:
# Load catalogues and merge to one
ds = pd.read_feather(f'{path}/input/starcat_GaiaDR3_LOPS2.ftr')
dn = pd.read_feather(f'{path}/input/starcat_GaiaDR3_LOPN1.ftr')
df = pd.concat([ds, dn])

# Number of objects
df.shape[0]

In [ ]:
# Start make some general cuts
dt0 = df[(df.class_name == 'AGN') & 
         (df. p_quasar > 0.999) & 
         (df.z > 0.09) & 
         (df.z_err < 0.05)]

# Sort after P magnitude
dt0 = dt0.sort_values(by=['ncams', 'Pmag'])

# Convert to percentage
dt0.p_quasar *= 100

# Number of objects
dt0.shape[0]

In [ ]:
# Define decontamination line
G_line = np.linspace(5, 19, 100)
mu_line = 10**(0.4*(G_line - 18.25))

# Mark outliers
mu = 10**(0.4*(dt0.Gmag - 18.25))
do1 = dt0[(dt0.pm > mu)]
dt1 = dt0[(dt0.pm < mu)]

# Number of objects
dt1.shape[0]

# Plot decontamination
fig = plt.figure(figsize=(6,4))
plt.plot(dt1.Gmag, dt1.pm, '.', c='k', alpha=0.1)
plt.plot(do1.Gmag, do1.pm, '.', c='r', alpha=0.8)
plt.plot(G_line, mu_line, '--', c='royalblue')
plt.xlabel(r'Gaia magnitude, $G$')
plt.ylabel(r'Proper motion, $\mu$ [mas/yr]')
plt.yscale('log')
plt.xlim(min(dt1.Gmag)-0.1, max(dt1.Gmag)+0.1)
plt.ylim(1e-3, 2e0)
plt.tight_layout();

In [ ]:
# Remove sources below black line above
do3 = dt2
def y(x): return x 
dt3 = dt2[(10**dt2.qso_non > 2.5) & (dt2.qso_non > y(dt2.qso_var))]

# Number of objects
dt3.shape[0]

# Check how many of the AGNs that has a confirmed host galaxy
dh3 = dt3[(dt3.host_galaxy == True)]

# All quasar above z > 3
dz3 = dt3[dt3.z > 3]

# Plot diagnostic diagram of Butler and Bloom (2011)
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.plot(10**do3.qso_var, 10**do3.qso_non, '*', c='k',        alpha=0.1, ms=5, label=f'All QSOs: N = {do3.shape[0]}')
ax.plot(10**dt3.qso_var, 10**dt3.qso_non, 'v', c='darkblue', alpha=0.2, ms=5, label=f'Cut QSOs: N = {dt3.shape[0]}')
ax.plot(10**dh3.qso_var, 10**dh3.qso_non, '.', c='orange',   alpha=0.9,       label=f'Host galaxy: N = {dh3.shape[0]}')
ax.plot(10**dz3.qso_var, 10**dz3.qso_non, 's', c='deeppink', alpha=0.7, ms=3, label=r'QSOs of $z>3$: ' + f'N = {dz3.shape[0]}')
ax.plot([0, 2.5], [1, 2.5],         '--', c='k', label=r'Cut: 3.5$\sigma$ significance')
ax.plot([2.5, 90], [y(2.5), y(90)], '-.', c='k', label='Cut: equal probability')
ax.set_xlabel(r'QSO variability, $\chi_{\rm QSO}^2$')
ax.set_ylabel(r'QSO non-variability, $\chi_{\rm False}^2$')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(5e-2, 5e2)
ax.set_ylim(0.9, 100)
ax.legend()
plt.tight_layout();
# fig.savefig(f'{fdir}/variability_QSO.png', bbox_inches='tight', dpi=300)

In [ ]:
# Load Quaia catalogue
filename = f'{path}/quaia_G20.5.fits'
data = Table.read(filename, format='fits')
df_quaia = data.to_pandas()
df_quaia.head()

In [ ]:
# Match catalogues
dx4 = df_quaia
dt4 = dt3[dt3['source_gaia_dr3'].isin(dx4['source_id'])].sort_values(by=['source_gaia_dr3']).reset_index(drop=True)
dx4 = dx4[dx4['source_id'].isin(dt4['source_gaia_dr3'])].sort_values(by=['source_id']).reset_index(drop=True)

# Insert relevant Quaia columns
dt4['z_quaia']     = dx4.redshift_quaia
dt4['z_quaia_err'] = dx4.redshift_quaia_err

# Sort after P magnitude
dt4 = dt4.sort_values(by=['ncams', 'Gmag'])
dt = dt4

# Seperate camera visibility
dt06 = dt4[dt4.ncams ==  6]
dt12 = dt4[dt4.ncams == 12]
dt18 = dt4[dt4.ncams == 18]
dt24 = dt4[dt4.ncams == 24]

# Sperate LOPS2 and LOPN1
ds4 = dt4[dt4.b < 0]
dn4 = dt4[dt4.b > 0]

# Total number targets
dt.shape[0]

In [ ]:
# Plot for thesis: investigate the discrepancy in redshifts
fig = plt.figure(figsize=(8,2.8*3))

ax1 = plt.subplot(3, 1, 1)
G_min, G_max = dt3.Gmag.min(), dt3.Gmag.max()
ax1.hist(dt06.Gmag, bins=N, range=(G_min, G_max), histtype='step', ec=c[0], lw=lw, label='6')
ax1.hist(dt12.Gmag, bins=N, range=(G_min, G_max), histtype='step', ec=c[1], lw=lw, label='12')
ax1.hist(dt18.Gmag, bins=N, range=(G_min, G_max), histtype='step', ec=c[2], lw=lw, label='18')
ax1.hist(dt24.Gmag, bins=N, range=(G_min, G_max), histtype='step', ec=c[3], lw=lw, label='24')
ax1.hist(dt3.Gmag,  bins=N, range=(G_min, G_max), histtype='step', ec=c[4], lw=lw, label='All')
ax1.set_xlabel(r'\textit{Gaia} magnitude, $G$')
ax1.set_xlim(G_min, 19)
ax1.set_ylim(0.8, 1000)
ax1.legend(loc='upper left', bbox_to_anchor=(0.06, 1.4), ncols=5)
ax1.set_yscale('log')

ax2 = plt.subplot(3, 1, 2)
z_min, z_max = dt3.z.min(), dt3.z.max()
ax2.hist(dt06.z, bins=N, range=(z_min, z_max), histtype='step', ec=c[0], lw=lw)
ax2.hist(dt12.z, bins=N, range=(z_min, z_max), histtype='step', ec=c[1], lw=lw)
ax2.hist(dt18.z, bins=N, range=(z_min, z_max), histtype='step', ec=c[2], lw=lw)
ax2.hist(dt24.z, bins=N, range=(z_min, z_max), histtype='step', ec=c[3], lw=lw)
ax2.hist(dt3.z,  bins=N, range=(z_min, z_max), histtype='step', ec=c[4], lw=lw)
ax2.set_xlabel(r'Redshift, $z$')
ax2.set_xlim(0, z_max)
ax2.set_yscale('log')

z_break = 2.8
ax3 = plt.subplot(3, 2, 5)
ax3.hist(dx3.redshift_quaia, bins=N, range=(z_min, z_max), histtype='step', ec='deeppink', lw=lw)
ax3.hist(dt3.z, bins=aN, range=(z_min, z_max), histtype='step', ec='k', lw=lw)
ax3.set_xlabel(r'Redshift, $z$')
ax3.set_xlim(0, z_break)

ax4 = plt.subplot(3, 2, 6)
ax4.hist(dx3.redshift_quaia, bins=N, range=(z_min, z_max), histtype='step', ec='deeppink', lw=lw, label='Quaia')
ax4.hist(dt3.z, bins=N, range=(z_min, z_max), histtype='step', ec='k', lw=lw, label=r'All (\textit{Gaia} DR3)')
ax4.set_xlabel(r'Redshift, $z$')
ax4.set_xlim(z_break, z_max)
ax4.set_ylim(0, 50)
ax4.legend(loc='upper right')

for ax in [ax1, ax2, ax3]:
    ax.set_ylabel('Count')
    if ax == ax3:
        ax.get_yaxis().set_label_coords(-0.12, 0.5)
    else:
        ax.get_yaxis().set_label_coords(-0.06, 0.5)
plt.tight_layout()
# fig.savefig(f'{fdir}/historgram_quasars_discrepancy.png', bbox_inches='tight', dpi=300)

### Post-process a single camera light curve

In [ ]:
# Load light curves of a single camera
lcs = LightCurve(odir, mode="multi")
files = lcs.files('hdf5', group=1, camera=1)

In [ ]:
# Merge all quarters
df0 = pd.DataFrame()
df1 = pd.DataFrame()
for f in files:
    lc = LightCurve(f)
    df1['time'] = lc.time()
    df1['flux'] = lc.flux(unit='norm')
    df0 = pd.concat([df0, df1])
    
# Deep copy of base flux
df0['flux_base'] = df0.flux

# Stitch and detrend the light curves
lc = LightCurve(df0, mode="multi")
df = lc.stitch(method='lowess', segment=10)
df = lc.detrend(column='flux_stitch', model='poly', poly_degree=1)

# Prepare for plot
dex = [13004, 26009, 39014, 52019, 65024, 78029, 91034, 104039, 117044, 130049, 143054]
df0['flux_base_filt'] = median_filter(df0.flux_base, 240)
flux_detrend = (df.flux_detrend-1)*1e6
flux_detrend_filt = median_filter(flux_detrend, 240)
time = df.time/86400

In [ ]:
# Produce combined post-processing figure
fig, ax = plt.subplots(3, 1, figsize=(9.5, 9.0))

ax[0].plot(time, df0.flux_base, '.', c='k', ms=1, alpha=0.2, label='Raw')
ax[0].plot(time, df0.flux_base_filt, '-', c='deeppink', lw=0.5, label="1d median")
for i in dex: ax[0].axvline(x=time.iloc[i], c='k', linestyle=':', lw=1)
ax[0].set_ylabel('Flux [norm]')

ax[1].plot(time, df.flux_stitch, '.', c='k', ms=1, alpha=0.2, label='Stitched')
for i in dex: ax[1].axvline(x=time.iloc[i], c='k', linestyle=':', lw=1)
ax[1].plot(time, df.flux_trend, '-', c='orange', lw=1.5, alpha=1, label='Trend')
ax[1].set_ylabel('Flux [norm]')

ax[2].plot(time, flux_detrend, '.', c='k', ms=1, alpha=0.2, label='Detrended')
ax[2].plot(time, flux_detrend_filt, '-', c='royalblue', lw=0.5, label="1d median")
# ax[2].plot(dv.time, dv.flux, '-', c='lime', lw=1)
ax[2].set_ylabel('Flux [ppt]')
ax[2].set_xlabel('Time [day]')
ax[2].set_ylim(-200, 250)

for i in range(3):
    ax[i].set_xlim(0, time.max())
    ax[i].legend(loc='upper right', ncol=3)
    ax[i].get_yaxis().set_label_coords(-0.07, 0.5)
plt.tight_layout(h_pad=0.1);

# Save figure
# fig.savefig(f'{fdir}/lc_pipeline.png', bbox_inches='tight', dpi=300)

## Spikey 

In [ ]:
# Load variable source file and add flux column [ppt]
varfile = path / 'varsource/varsource_spikey_fiducial.txt'
dv = pd.read_csv(varfile, sep=' ', names=['time', 'dmag'])
dv['flux'] = (10**(-dv.dmag/2.5) - 1) * 1e3
dv['time'] /= 86400

In [ ]:
# Light curve of Spikey
lcs = LightCurve(sdir / 'thesis/spikey', mode="multi")
df = merge(lcs)
time = df.time / 86400

In [ ]:
# Plot light curve of Spikey
fig, (ax0, ax1) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [2, 1], 'wspace': 0.04}, figsize=(9.5, 3.5))

ax0.plot(time, df.flux, '.', c='k', ms=3, alpha=0.3, label=r'24 N-CAMs')
ax0.plot(time, df.fmed, '-', c='royalblue', lw=0.5, label='1d median')
ax0.plot(dv.time, dv.flux, '-', c='lime', lw=1.3, label='Input model')
ax0.set_ylabel('Flux [ppt]')
ax0.set_xlim(0, time.max())
ax0.set_ylim(-180, 190)
ax0.legend(loc='upper right', ncol=3, bbox_to_anchor=(1.4, 1.2))

ax1.plot(time, df.flux, 'o', c='k', ms=3, alpha=0.3, label=r'Spikey observed with 24 N-CAMs')
ax1.plot(time, df.fmed, '-', c='royalblue', lw=1.3)
ax1.plot(dv.time, dv.flux, '-', c='lime', lw=1.3)
ax1.set_xlim(300, 350)
ax1.set_ylim(-180, 190)
ax1.set_xticks([310, 330, 350])
ax1.set_yticklabels([])

plt.tight_layout()
fig.text(0.5, -0.05, 'Time [days]', ha='center')

# Save figure
# fig.savefig(f'{fdir}/lc_spikey.png', bbox_inches='tight', dpi=300)

### Spikey at different magnitudes

In [ ]:
# Load
lcs_mag16 = LightCurve(sdir / 'thesis/000000001_mag16', mode="multi")
lcs_mag17 = LightCurve(sdir / 'thesis/000000001_mag17', mode="multi")
lcs_mag18 = LightCurve(sdir / 'thesis/000000001_mag18', mode="multi")
lcs_mag19 = LightCurve(sdir / 'thesis/000000001_mag19', mode="multi")

# Reduce light curves
df_mag16 = merge(lcs_mag16)
df_mag17 = merge(lcs_mag17)
df_mag18 = merge(lcs_mag18)
df_mag19 = merge(lcs_mag19)

# Single time array
time = df_mag16.time / 86400

In [ ]:
# Plot magnitude dependence of Spikey
fig, ax = plt.subplots(4, 1, figsize=(9.5, 12.8))

ax[0].plot(time, df_mag16.flux, '.', c='k', ms=2, alpha=0.3, label=r'$G=16$')
ax[0].plot(time, df_mag16.fmed, '-', c='royalblue', lw=0.5)
ax[0].plot(dv.time, dv.flux, '-', c='lime', lw=1)
ax[0].set_ylabel('Flux [ppt]')

ax[1].plot(time, df_mag17.flux, '.', c='k', ms=2, alpha=0.3, label=r'$G=17$')
ax[1].plot(time, df_mag17.fmed, '-', c='royalblue', lw=0.5)
ax[1].plot(dv.time, dv.flux, '-', c='lime', lw=1)
ax[1].set_ylabel('Flux [ppt]')

ax[2].plot(time, df_mag18.flux, '.', c='k', ms=2, alpha=0.3, label=r'$G=18$')
ax[2].plot(time, df_mag18.fmed, '-', c='royalblue', lw=0.5)
ax[2].plot(dv.time, dv.flux, '-', c='lime', lw=1)
ax[2].set_ylabel('Flux [ppt]')

ax[3].plot(time, df_mag19.flux, '.', c='k', ms=2, alpha=0.3, label=r'$G=19$')
ax[3].plot(time, df_mag19.fmed, '-', c='royalblue', lw=0.5)
ax[3].plot(dv.time, dv.flux, '-', c='lime', lw=1)
ax[3].set_ylabel('Flux [ppt]')
ax[3].set_xlabel('Time [day]')

for i in range(4):
    ax[i].set_xlim(0, time.max())
    ax[i].legend(loc='upper right')
    ax[i].get_yaxis().set_label_coords(-0.07, 0.5)
plt.tight_layout(h_pad=0.1);

# Save figure
# fig.savefig(f'{fdir}/lc_magnitudes.png', bbox_inches='tight', dpi=300)

In [ ]:
# def merge(lcs):
#     # Combine the two datasets while reducing each
#     dx = pd.DataFrame()
#     for g,c in zip(tqdm(range(1,5), bar_format=ut.tqdmBar()), range(1,7)):

#         files = lcs.files('hdf5', group=g, camera=c)

#         # Merge all quarters
#         df0 = pd.DataFrame()
#         df1 = pd.DataFrame()
#         for f in files:
#             lc = LightCurve(f)
#             df1['time'] = lc.time()
#             df1['flux'] = lc.flux(unit='norm')
#             df0 = pd.concat([df0, df1])

#         # Deep copy of base flux
#         df0['flux_base'] = df0.flux

#         # Stitching
#         lc = LightCurve(df0, mode="multi")
#         df = lc.stitch(method='lowess', segment=5)

#         # Detrending
#         df = lc.detrend(column='flux_stitch', model='poly', poly_degree=1)
#         df.flux_detrend = (df.flux_detrend-1)*1e6

#         # Add median filter [1 h]
#         df['flux_detrend_med'] = median_filter(df.flux_detrend, 144)
    
#         # Merge light curve
#         dx = pd.concat([dx, df])

#     # Group after timings 
#     dx = dx.groupby('time').mean().reset_index()
    
#     # Bin data [1 h]
#     tbin = int(3600)
#     tdur = dx.time.iloc[-1] - dx.time.iloc[0]
#     bins = int(tdur/tbin)
#     flux_bin, time_bin, _= binned_statistic(dx.time, dx.flux_detrend, 'median', bins)
#     flux_med, time_bin, _= binned_statistic(dx.time, dx.flux_detrend_med, 'median', bins)
#     time_bin = time_bin[:-1] + np.diff(time_bin)[0]/2.
#     dx = pd.DataFrame({'time':time_bin, 'flux':flux_bin, 'fmed':flux_med})
    
#     return dx